# LeetCode 2311 — Longest Binary Subsequence Less Than or Equal to K bằng Python

Link đề: https://leetcode.com/problems/longest-binary-subsequence-less-than-or-equal-to-k/ · Độ khó: **Medium** · Chủ đề: String, Dynamic Programming, Greedy, Memoization

Skeleton LeetCode:

```python
class Solution:
    def longestSubsequence(self, s: str, k: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra: chữ số `0` **không bao giờ làm tăng** giá trị (chỉ đẩy các bit 1 lên cao hơn nếu đứng **sau** chúng) — giữ mọi số 0 ở **bên trái** các bit 1 được chọn là "miễn phí".
2. Tham lam từ **phải sang trái**: chiều dài hiện tại chính là **vị trí bit** của ký tự tiếp theo; nhận bit 1 nếu tổng vẫn ≤ `k`.
3. Chứng minh tham lam tương đương "**hậu tố dài nhất có giá trị ≤ k** + mọi số 0 đứng trước nó", và rút ra công thức O(n) theo `k.bit_length()`.
4. Đối chiếu với DP O(n²) "giá trị nhỏ nhất cho mỗi độ dài" và vét cạn 2ⁿ.
5. Cẩn thận với giới hạn bit: `k ≤ 10⁹ < 2³⁰` ⇒ bit 1 ở vị trí ≥ 30 không bao giờ chọn được.

## 2. Đề bài & ví dụ

Cho chuỗi nhị phân `s` và số nguyên dương `k`. Trả về độ dài của **dãy con dài nhất** của `s` mà biểu diễn một số nhị phân **≤ `k`**. Dãy con có thể có số 0 đứng đầu; chuỗi rỗng có giá trị 0.

**Ví dụ 1:** `s = "1001010"`, `k = 5` → `5` (ví dụ `"00010"` = 2; `"00100"` = 4, `"00101"` = 5 cũng được).

**Ví dụ 2:** `s = "00101001"`, `k = 1` → `6` (`"000001"` = 1).

**Ràng buộc:** `1 ≤ len(s) ≤ 1000`; `1 ≤ k ≤ 10⁹`.

In [1]:
from functools import lru_cache
from itertools import combinations
from math import inf
import random, time

examples = [("1001010", 5, 5), ("00101001", 1, 6)]
for s, k, exp in examples:
    print(f"s={s!r} k={k} (nhị phân {bin(k)[2:]}) -> kỳ vọng {exp}")

s='1001010' k=5 (nhị phân 101) -> kỳ vọng 5
s='00101001' k=1 (nhị phân 1) -> kỳ vọng 6


## 3. Kiến thức nền

### 3.1 Giá trị của chuỗi nhị phân và vai trò của vị trí

`value(b_{L−1} … b₁ b₀) = Σ bᵢ · 2ⁱ` — bit ở **cuối** chuỗi có trọng số nhỏ nhất. Số 0 đứng đầu không đổi giá trị.

In [2]:
for t in ["00010", "00100", "00101", "000001", ""]:
    print(repr(t), "=", int(t, 2) if t else 0)

'00010' = 2
'00100' = 4
'00101' = 5
'000001' = 1
'' = 0


### 3.2 Chèn một ký tự vào **đầu** một dãy con đã có

Nếu đã có dãy con `T` dài `L` và chèn `c` vào trước: giá trị mới = `c · 2^L + value(T)`. Chèn `'0'` không đổi giá trị; chèn `'1'` cộng `2^L`. Vì vậy khi xây dãy con **từ phải sang trái**, độ dài hiện tại `L` chính là vị trí bit của ký tự sắp chèn.

In [3]:
T = "010"; L = len(T)
print("value(T) =", int(T, 2), "| chèn '0':", int("0" + T, 2), "| chèn '1':", int("1" + T, 2), "= 2^L + value(T) =", 2 ** L + int(T, 2))

value(T) = 2 | chèn '0': 2 | chèn '1': 10 = 2^L + value(T) = 10


### 3.3 `int.bit_length()`

`k.bit_length()` = số bit cần để biểu diễn `k` (`k ≥ 1`). Mọi số có `≥ m + 1` bit có nghĩa đều `≥ 2^m > k` với `m = k.bit_length()`.

In [4]:
for k in [1, 5, 10**9]:
    print(k, bin(k), "bit_length =", k.bit_length(), "| 2^bit_length =", 2 ** k.bit_length(), "> k")

1 0b1 bit_length = 1 | 2^bit_length = 2 > k
5 0b101 bit_length = 3 | 2^bit_length = 8 > k
1000000000 0b111011100110101100101000000000 bit_length = 30 | 2^bit_length = 1073741824 > k


## 4. Ý tưởng nền tảng / trực giác

**Quan sát 1 — số 0 là miễn phí khi ở bên trái.** Lấy một dãy con hợp lệ bất kỳ; thêm mọi số 0 của `s` nằm **trước** bit 1 đầu tiên của nó (hoặc tất cả, nếu không có bit 1) không làm tăng giá trị. Thậm chí, thêm **mọi** số 0 của `s` vào bên trái mọi bit 1 không chắc giữ được — vì một số 0 nằm **giữa** hai bit 1 sẽ đẩy bit 1 bên trái lên cao hơn. Nhưng nếu ta xây từ phải sang trái và luôn nhận số 0, thì số 0 chỉ làm tăng **vị trí** của các bit 1 sau này — điều đó được tính đúng vào chi phí.

**Quan sát 2 — bit 1 càng ở bên phải càng rẻ.** Duyệt từ phải sang trái, giữ `L` = độ dài hiện tại, `v` = giá trị hiện tại:
- gặp `'0'`: nhận luôn (`L += 1`, `v` không đổi);
- gặp `'1'`: nhận **nếu** `v + 2^L ≤ k`.

Tính tay ví dụ 1 (`s = "1001010"`, `k = 5`), duyệt từ phải:

| ký tự (từ phải) | L trước | thử | v sau | L sau |
|---|---|---|---|---|
| 0 | 0 | nhận 0 | 0 | 1 |
| 1 | 1 | 0 + 2¹ = 2 ≤ 5 ✔ | 2 | 2 |
| 0 | 2 | nhận 0 | 2 | 3 |
| 1 | 3 | 2 + 2³ = 10 > 5 ✘ | 2 | 3 |
| 0 | 3 | nhận 0 | 2 | 4 |
| 0 | 4 | nhận 0 | 2 | 5 |
| 1 | 5 | 2 + 32 > 5 ✘ | 2 | 5 |

⇒ `5`, dãy con `"00010"` ✔.

In [5]:
def greedy_trace(s, k):
    L = v = 0; chosen = []
    for i in range(len(s) - 1, -1, -1):
        c = s[i]
        if c == "0":
            chosen.append(i); L += 1; note = "nhận 0"
        elif v + (1 << L) <= k:
            v += 1 << L; chosen.append(i); L += 1; note = f"nhận 1 (v = {v})"
        else:
            note = f"bỏ 1 (v + 2^{L} = {v + (1 << L)} > {k})"
        print(f"  s[{i}] = {c}: {note:<28} L = {L}")
    sub = "".join(s[i] for i in sorted(chosen))
    print("  dãy con:", repr(sub), "=", int(sub, 2) if sub else 0)
    return L

print("đáp án:", greedy_trace(*examples[0][:2]))

  s[6] = 0: nhận 0                       L = 1
  s[5] = 1: nhận 1 (v = 2)               L = 2
  s[4] = 0: nhận 0                       L = 3
  s[3] = 1: bỏ 1 (v + 2^3 = 10 > 5)      L = 3
  s[2] = 0: nhận 0                       L = 4
  s[1] = 0: nhận 0                       L = 5
  s[0] = 1: bỏ 1 (v + 2^5 = 34 > 5)      L = 5
  dãy con: '00010' = 2
đáp án: 5


## 5. Cách 1 — Brute Force / baseline

### 5.1 Duyệt mọi dãy con — O(2ⁿ · n)

In [6]:
def brute(s, k):
    n = len(s)
    for L in range(n, -1, -1):
        for idx in combinations(range(n), L):
            t = "".join(s[i] for i in idx)
            if (int(t, 2) if t else 0) <= k: return L
    return 0

for s, k, exp in examples:
    print(brute(s, k), "OK" if brute(s, k) == exp else "SAI")

5 OK
6 OK


`n = 1000` ⇒ `2¹⁰⁰⁰` — chỉ dùng kiểm chứng với `n ≤ 14`.

### 5.2 Baseline 2 — DP "giá trị nhỏ nhất cho mỗi độ dài" — O(n²)

`best[L]` = giá trị **nhỏ nhất** của một dãy con dài `L` trong tiền tố đã xét. Duyệt trái → phải; thêm ký tự `c` vào **cuối**: giá trị mới `= 2 · old + c`. Đáp án = `L` lớn nhất có `best[L] ≤ k`.

In [7]:
def dp_min_value(s, k):
    n = len(s); best = [0] + [inf] * n
    for c in s:
        b = int(c)
        for L in range(n - 1, -1, -1):              # duyệt ngược: mỗi ký tự dùng một lần
            if best[L] != inf:
                cand = best[L] * 2 + b
                if cand < best[L + 1]: best[L + 1] = cand
    return max(L for L in range(n + 1) if best[L] <= k)

for s, k, exp in examples:
    print(dp_min_value(s, k), "OK" if dp_min_value(s, k) == exp else "SAI")

5 OK
6 OK


DP đúng vì giá trị mới chỉ phụ thuộc giá trị cũ một cách **đơn điệu** (`2x + b` tăng theo `x`) ⇒ chỉ cần nhớ giá trị nhỏ nhất cho mỗi độ dài. Nhưng `best[L]` có thể là số **hàng trăm bit** (Python vẫn tính được, chậm hơn), và O(n²) = `10⁶` phép trên số lớn. **Nút thắt:** ta đang theo dõi quá nhiều; tham lam chỉ cần một trạng thái.

## 6. Ý tưởng thuật toán / bước đột phá

**Tham lam từ phải (doocs):**

```
L = v = 0
for c in reversed(s):
    if c == '0': L += 1
    elif L < 30 and v + (1 << L) <= k: v += 1 << L; L += 1
return L
```

`L < 30` vì `2³⁰ > 10⁹ ≥ k` — tránh dịch bit vô ích (Python không tràn số, nhưng đây là điều kiện bắt buộc trong C++/Java).

**Dạng đóng tương đương (O(n), không cần vòng phức tạp):** gọi `m = k.bit_length()`.
- Nếu `n ≤ m`: xét toàn chuỗi như hậu tố.
- Hậu tố dài `m` của `s` có giá trị `< 2^m`; nếu nó `≤ k` thì nhận nguyên hậu tố, ngược lại nhận hậu tố dài `m − 1` (luôn `< 2^{m−1} ≤ k`).
- Cộng thêm **số chữ số 0** trong phần còn lại bên trái.

Mục 10 chứng minh tham lam = "hậu tố dài nhất có giá trị ≤ k + mọi số 0 trước nó" = dạng đóng.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tham lam từ phải (cách doocs)

In [8]:
def greedy_right(s, k):
    ans = v = 0
    for c in s[::-1]:
        if c == "0":
            ans += 1
        elif ans < 30 and (v | 1 << ans) <= k:
            v |= 1 << ans
            ans += 1
    return ans

for s, k, exp in examples:
    print(greedy_right(s, k), "OK" if greedy_right(s, k) == exp else "SAI")

5 OK
6 OK


`v | 1 << ans` bằng `v + (1 << ans)` vì bit `ans` của `v` chắc chắn đang là 0 (mọi bit đã đặt đều ở vị trí `< ans`).

### 7.2 Cách 3 — Dạng đóng theo `k.bit_length()`

In [9]:
def closed_form(s, k):
    n, m = len(s), k.bit_length()
    if n <= m:
        return n if int(s, 2) <= k else n - 1      # cả chuỗi là "hậu tố"; nếu > k thì bỏ bit đầu (là '1')
    tail = s[n - m:]
    take = m if int(tail, 2) <= k else m - 1
    return s[:n - m].count("0") + take

for s, k, exp in examples:
    print(closed_form(s, k), "OK" if closed_form(s, k) == exp else "SAI")

5 OK
6 OK


Trường hợp `n ≤ m` cần cẩn thận: nếu cả chuỗi `≤ k` thì đáp án `n`. Nếu không, `s` có đúng `m` ký tự (vì `n < m` thì `value < 2^n ≤ 2^{m−1} ≤ k`), và ta cần dãy con dài `n − 1`: bỏ **một** ký tự — bỏ ký tự đầu (là `'1'`, vì `value > k ≥ 2^{m−1}` nên bit đầu là 1) cho hậu tố dài `m − 1` luôn `≤ k`. ⇒ `n − 1`. Cách 4 dưới đây là phiên bản tường minh của cùng ý tưởng, không cần phân trường hợp.

### 7.3 Cách 4 — Hậu tố dài nhất ≤ k + đếm số 0 (dạng tường minh)

In [10]:
def suffix_plus_zeros(s, k):
    n = len(s); L = 0; v = 0
    while L < n and L < 31:                       # tìm hậu tố dài nhất có giá trị <= k
        nv = v + (int(s[n - 1 - L]) << L)
        if nv > k: break
        v = nv; L += 1
    return L + s[:n - L].count("0")

for s, k, exp in examples:
    print(suffix_plus_zeros(s, k), "OK" if suffix_plus_zeros(s, k) == exp else "SAI")

5 OK
6 OK


Lưu ý: hậu tố dài nhất `≤ k` được tìm bằng cách mở rộng sang trái tới khi vượt `k`. Vì giá trị hậu tố **không giảm** khi mở rộng (thêm bit 0 không đổi, thêm bit 1 tăng), điểm dừng đầu tiên là đúng. Sau khi dừng, ta không nhận thêm bit 1 nào nữa, nhưng vẫn nhận mọi số 0 bên trái — chính là `s[:n − L].count("0")`.

### 7.4 Cách 5 — Đệ quy có nhớ theo (vị trí từ phải, độ dài hiện tại) — kiểm chứng tham lam

In [11]:
def memo(s, k):
    n = len(s)
    @lru_cache(maxsize=None)
    def go(i, L, v):                              # xét s[i] (từ phải), đã có dãy con dài L giá trị v
        if i < 0: return L
        best = go(i - 1, L, v)                     # bỏ s[i]
        nv = v + (int(s[i]) << L)
        if nv <= k: best = max(best, go(i - 1, L + 1, nv))
        return best
    r = go(n - 1, 0, 0); go.cache_clear()
    return r

for s, k, exp in examples:
    print(memo(s, k), "OK" if memo(s, k) == exp else "SAI")

5 OK
6 OK


Không gian trạng thái `(i, L, v)` có thể lớn (v tới `k`) — chỉ dùng cho chuỗi ngắn.

## 8. TRACE từng bước

### 8.1 Trace tham lam trên ví dụ 2

In [12]:
print("đáp án:", greedy_trace(*examples[1][:2]))

  s[7] = 1: nhận 1 (v = 1)               L = 1
  s[6] = 0: nhận 0                       L = 2
  s[5] = 0: nhận 0                       L = 3
  s[4] = 1: bỏ 1 (v + 2^3 = 9 > 1)       L = 3
  s[3] = 0: nhận 0                       L = 4
  s[2] = 1: bỏ 1 (v + 2^4 = 17 > 1)      L = 4
  s[1] = 0: nhận 0                       L = 5
  s[0] = 0: nhận 0                       L = 6
  dãy con: '000001' = 1
đáp án: 6


`k = 1` nên chỉ bit 1 ở **vị trí 0** (ký tự cuối) có thể được nhận. Ký tự cuối của `s` là `'1'` ⇒ nhận ngay. Mọi bit 1 khác bị bỏ; mọi số 0 được nhận ⇒ `1 + 5 = 6`.

### 8.2 Trace dạng đóng

In [13]:
for s, k, _ in examples + [("1111", 3, None), ("10", 1, None)]:
    n, m = len(s), k.bit_length()
    if n > m:
        tail = s[n - m:]
        print(f"s={s!r} k={k} m={m}: hậu tố {tail!r} = {int(tail, 2)} {'<=' if int(tail, 2) <= k else '>'} k -> lấy {m if int(tail, 2) <= k else m - 1} ký tự cuối + {s[:n - m].count('0')} số 0 bên trái = {closed_form(s, k)}")
    else:
        print(f"s={s!r} k={k} m={m}: n <= m -> {closed_form(s, k)}")

s='1001010' k=5 m=3: hậu tố '010' = 2 <= k -> lấy 3 ký tự cuối + 2 số 0 bên trái = 5
s='00101001' k=1 m=1: hậu tố '1' = 1 <= k -> lấy 1 ký tự cuối + 5 số 0 bên trái = 6
s='1111' k=3 m=2: hậu tố '11' = 3 <= k -> lấy 2 ký tự cuối + 0 số 0 bên trái = 2
s='10' k=1 m=1: hậu tố '0' = 0 <= k -> lấy 1 ký tự cuối + 0 số 0 bên trái = 1


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def longestSubsequence(self, s: str, k: int) -> int:
        length = 0                                 # độ dài dãy con = vị trí bit của ký tự kế tiếp
        value = 0
        for c in reversed(s):
            if c == "0":
                length += 1                        # số 0 không làm tăng giá trị
            elif length < 31 and value + (1 << length) <= k:
                value += 1 << length               # bit 1 ở vị trí thấp nhất có thể
                length += 1
        return length

In [15]:
sol = Solution()
for s, k, exp in examples:
    got = sol.longestSubsequence(s, k)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

5 | kỳ vọng 5 | OK
6 | kỳ vọng 6 | OK


## 10. Vì sao thuật toán đúng

Gọi `n = len(s)`, `Z(j)` = số chữ số 0 trong `s[0..j−1]`. Lấy một dãy con hợp lệ (giá trị ≤ k) **dài nhất** `T`; ta biến đổi nó, không làm giảm độ dài và không làm giá trị vượt `k`, cho tới khi nó có đúng dạng mà tham lam tạo ra.

**Bước 1 — có thể giả sử `T` chứa mọi số 0 của `s`.** Giả sử `T` bỏ một số 0 ở chỉ số `q`.
- Nếu mọi bit 1 của `T` nằm **sau** `q` (hoặc `T` không có bit 1): thêm số 0 đó vào là số 0 đứng đầu ⇒ giá trị không đổi, độ dài tăng — mâu thuẫn với "dài nhất". Vậy trường hợp này không xảy ra.
- Ngược lại, gọi `p` = bit 1 **đầu tiên** của `T` (`p < q`). Phần của `T` từ `p` trở đi là `"1" + R` với `R` dài `r − 1`, giá trị `2^{r−1} + val(R) ≤ k`. Bỏ bit 1 ở `p` và thêm số 0 ở `q`: phần mới là `R` với một số 0 chèn vào giữa — chèn một số 0 vào một chuỗi chỉ có thể **nhân đôi** các bit bên trái chỗ chèn, nên giá trị mới `≤ 2·val(R) ≤ 2^{r−1} + val(R) ≤ k` (vì `val(R) < 2^{r−1}`). Độ dài không đổi, số số 0 trong `T` tăng 1.

Lặp lại (số số 0 tăng nghiêm ngặt) ⇒ tồn tại `T` dài nhất chứa mọi số 0.

**Bước 2 — có thể giả sử các bit 1 của `T` là "đóng về bên phải".** Giả sử `T` chọn bit 1 ở `p` nhưng bỏ bit 1 ở `q > p`. Viết phần của `T` từ `p` là `"1" + X + Y` với `X` = các ký tự đã chọn giữa `p` và `q` (dài `a`), `Y` = các ký tự đã chọn sau `q` (dài `b`). Đổi: bỏ `p`, chọn `q` ⇒ phần mới là `X + "1" + Y`. Chênh lệch giá trị:
`(X·2^{b+1} + 2^b + Y) − (2^{a+b} + X·2^b + Y) = 2^b·(X + 1 − 2^a) ≤ 0` vì `X < 2^a`.
Giá trị không tăng, độ dài không đổi. Lặp lại ⇒ tập bit 1 được chọn là **mọi** bit 1 từ một chỉ số `j` trở đi.

**Bước 3 — dạng cuối.** Sau hai bước, `T` = mọi số 0 của `s` + mọi bit 1 của `s[j..]`, tức là `T` = (các số 0 của `s[0..j−1]`) + **toàn bộ hậu tố** `s[j..]`, độ dài `Z(j) + (n − j)`, giá trị `= val(s[j..]) ≤ k`. Hàm `g(ℓ) = Z(n − ℓ) + ℓ` (với `ℓ = n − j`) **không giảm** theo `ℓ` (tăng `ℓ` thêm 1: `+1`, còn `Z` giảm tối đa 1), và `val(s[n−ℓ..])` không giảm theo `ℓ` ⇒ tối ưu là `ℓ` = độ dài **hậu tố dài nhất có giá trị ≤ k**.

**Tham lam tạo đúng nghiệm này.** Từ phải sang trái, tham lam nhận mọi ký tự tới bit 1 đầu tiên làm vượt `k` (đó là hậu tố dài nhất ≤ k); sau đó `v` không đổi còn `L` chỉ tăng, nên mọi bit 1 về sau cũng vượt `k` và bị bỏ; mọi số 0 đều được nhận. ⇒ Kết quả tham lam `= Z(n − ℓ) + ℓ` = tối ưu. ∎

**Dạng đóng.** Đặt `m = k.bit_length()`, `ℓ` = độ dài hậu tố dài nhất có giá trị ≤ k. (i) Mọi hậu tố dài `m − 1` có giá trị `< 2^{m−1} ≤ k` ⇒ `ℓ ≥ min(n, m − 1)`. (ii) Một hậu tố dài `ℓ > m` có giá trị ≤ k chỉ khi mọi ký tự ở vị trí bit `≥ m` của nó đều là 0 (một bit 1 ở vị trí `≥ m` đã cho giá trị `≥ 2^m > k`); khi đó các số 0 thêm vào đó làm `ℓ` tăng đúng bằng lượng `Z` giảm, nên `g(ℓ) = g(m)`. Vậy đáp án `= g(m)` nếu hậu tố dài `m` có giá trị ≤ k, ngược lại `g(m − 1)` — đúng công thức ở mục 7.2 (với `n ≤ m` xử lý riêng như đã nói).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi dãy con | O(2ⁿ · n) | O(n) |
| 1b. DP giá trị nhỏ nhất | O(n²) (số lớn) | O(n) số lớn |
| 2. Tham lam từ phải | O(n) | O(1) |
| 3. Dạng đóng | O(n) | O(1) |
| 4. Hậu tố + số 0 | O(n) | O(1) |
| 5. Đệ quy có nhớ | lớn (theo `k`) | lớn |

In [16]:
rng = random.Random(2311)
big = "".join(rng.choice("01") for _ in range(1000))
for k in [1, 10**3, 10**9]:
    res = []
    for name, f in [("dp_min_value", dp_min_value), ("greedy_right", greedy_right), ("closed_form", closed_form),
                    ("suffix_plus_zeros", suffix_plus_zeros), ("Solution", Solution().longestSubsequence)]:
        t0 = time.perf_counter(); res.append(f(big, k)); dt = time.perf_counter() - t0
        print(f"k={k:<11} {name:<18} {dt*1000:8.2f} ms -> {res[-1]}")
    assert len(set(res)) == 1

k=1           dp_min_value          61.56 ms -> 506
k=1           greedy_right           0.04 ms -> 506
k=1           closed_form            0.01 ms -> 506
k=1           suffix_plus_zeros      0.01 ms -> 506
k=1           Solution               0.03 ms -> 506


k=1000        dp_min_value          51.96 ms -> 512
k=1000        greedy_right           0.04 ms -> 512
k=1000        closed_form            0.01 ms -> 512
k=1000        suffix_plus_zeros      0.01 ms -> 512
k=1000        Solution               0.03 ms -> 512


k=1000000000  dp_min_value          59.62 ms -> 521
k=1000000000  greedy_right           0.03 ms -> 521
k=1000000000  closed_form            0.01 ms -> 521
k=1000000000  suffix_plus_zeros      0.01 ms -> 521
k=1000000000  Solution               0.03 ms -> 521


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [17]:
ALL = {"brute": brute, "dp_min_value": dp_min_value, "greedy_right": greedy_right, "closed_form": closed_form,
       "suffix_plus_zeros": suffix_plus_zeros, "memo": memo, "Solution": Solution().longestSubsequence}
edge = examples + [
    ("0", 1, 1), ("1", 1, 1), ("11", 1, 1), ("10", 1, 1),     # "10" = 2 > 1 nên chỉ dài 1
    ("1111", 3, 2), ("0000", 1, 4), ("111111", 1000, 6),
    ("1" * 12, 1, 1), ("10" * 6, 2, 7),
]
for s, k, exp in edge:
    ref = brute(s, k)
    for name, f in ALL.items():
        assert f(s, k) == ref, (name, s, k, f(s, k), ref)
    print(f"s={s!r:<16} k={k:<5} -> {ref} | ghi trong bảng: {exp} | khớp: {ref == exp}")

s='1001010'        k=5     -> 5 | ghi trong bảng: 5 | khớp: True
s='00101001'       k=1     -> 6 | ghi trong bảng: 6 | khớp: True
s='0'              k=1     -> 1 | ghi trong bảng: 1 | khớp: True
s='1'              k=1     -> 1 | ghi trong bảng: 1 | khớp: True
s='11'             k=1     -> 1 | ghi trong bảng: 1 | khớp: True
s='10'             k=1     -> 1 | ghi trong bảng: 1 | khớp: True
s='1111'           k=3     -> 2 | ghi trong bảng: 2 | khớp: True
s='0000'           k=1     -> 4 | ghi trong bảng: 4 | khớp: True
s='111111'         k=1000  -> 6 | ghi trong bảng: 6 | khớp: True
s='111111111111'   k=1     -> 1 | ghi trong bảng: 1 | khớp: True
s='101010101010'   k=2     -> 7 | ghi trong bảng: 7 | khớp: True


Case `"10"`, `k = 1`: cả chuỗi `"10"` = 2 > 1, nên dài nhất là 1 (`"1"` hoặc `"0"`). Dễ tính nhầm thành 2 nếu quên rằng số 0 ở **cuối** làm **nhân đôi** giá trị của bit 1 đứng trước — chính là lý do mọi test đều đối chiếu với vét cạn (cột `khớp:`).

### 12.2 Đối chiếu ngẫu nhiên với vét cạn (seed cố định)

In [18]:
rng = random.Random(42)
for _ in range(1500):
    n = rng.randint(1, 12); s = "".join(rng.choice("01") for _ in range(n))
    k = rng.choice([1, 2, 3, rng.randint(1, 64), rng.randint(1, 5000)])
    ref = brute(s, k)
    for name, f in ALL.items():
        assert f(s, k) == ref, (name, s, k)
print("1500 chuỗi ngẫu nhiên OK")

1500 chuỗi ngẫu nhiên OK


### 12.3 Tính chất: đáp án ≥ số chữ số 0; không giảm theo `k`; đáp án `= n` ⇔ `int(s, 2) ≤ k`

In [19]:
rng = random.Random(7)
for _ in range(1000):
    n = rng.randint(1, 60); s = "".join(rng.choice("01") for _ in range(n))
    ks = sorted(rng.randint(1, 10**9) for _ in range(5))
    vals = [Solution().longestSubsequence(s, k) for k in ks]
    assert all(v >= s.count("0") for v in vals) and vals == sorted(vals)
    for k, v in zip(ks, vals):
        assert (v == n) == (int(s, 2) <= k)
print("tính chất OK")

tính chất OK


### 12.4 Edge cực trị: `n = 1000`, `k = 10⁹`, bit 1 ở vị trí 29 và 30

In [20]:
s = "1" + "0" * 29                     # = 2^29 = 536870912 <= 10^9
assert Solution().longestSubsequence(s, 10**9) == 30
s = "1" + "0" * 30                     # = 2^30 > 10^9
assert Solution().longestSubsequence(s, 10**9) == 30
s = "1" * 1000
print(Solution().longestSubsequence(s, 10**9), "(tối đa số bit 1 liên tiếp ở cuối <= 10^9: 2^29 - 1 < 10^9 < 2^30 - 1)")
assert Solution().longestSubsequence(s, 10**9) == 29

29 (tối đa số bit 1 liên tiếp ở cuối <= 10^9: 2^29 - 1 < 10^9 < 2^30 - 1)


### 12.5 Dạng đóng khớp tham lam trên chuỗi dài và `k` lớn

In [21]:
rng = random.Random(125)
for _ in range(3000):
    n = rng.randint(1, 80); s = "".join(rng.choice("01" if rng.random() < 0.5 else "0001") for _ in range(n))
    k = rng.randint(1, 10**9)
    assert closed_form(s, k) == greedy_right(s, k) == suffix_plus_zeros(s, k) == dp_min_value(s, k)
print("3000 chuỗi dài tới 80, k tới 10^9: bốn cách khớp nhau")

3000 chuỗi dài tới 80, k tới 10^9: bốn cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam từ **trái** sang phải (nhận bit 1 sớm)

In [22]:
def wrong_left(s, k):
    t = ""
    for c in s:
        if int(t + c, 2) <= k: t += c              # SAI: bit 1 nhận sớm sẽ bị đẩy lên cao bởi các ký tự sau
    return len(t)

for s, k, _ in examples:
    print("bản sai :", wrong_left(s, k), "| bản đúng:", Solution().longestSubsequence(s, k))

bản sai : 3 | bản đúng: 5
bản sai : 3 | bản đúng: 6


**Vì sao sai:** ví dụ 1: nhận `'1'` đầu tiên (= 1), rồi `"10"` = 2, `"100"` = 4, `"1000"` = 8 > 5 bỏ… mỗi ký tự thêm vào **sau** nhân đôi giá trị bit 1 đã nhận ⇒ không thể nhận nhiều số 0 nữa. Xây từ phải sang trái thì giá trị các bit đã nhận **không đổi** khi thêm ký tự mới vào trước.

### Lỗi 2 — Giới hạn bit quá chặt (`L < 29`)

In [23]:
def wrong_cap(s, k):
    ans = v = 0
    for c in reversed(s):
        if c == "0": ans += 1
        elif ans < 29 and v + (1 << ans) <= k:      # SAI: 2^29 <= 10^9 vẫn có thể nhận
            v += 1 << ans; ans += 1
    return ans

s = "1" + "0" * 29
print("bản sai :", wrong_cap(s, 10**9), "| bản đúng:", Solution().longestSubsequence(s, 10**9))
print("ví dụ đề (trùng hợp):", [wrong_cap(s, k) for s, k, _ in examples])

bản sai : 29 | bản đúng: 30
ví dụ đề (trùng hợp): [5, 6]


**Vì sao sai:** `2²⁹ = 536 870 912 ≤ 10⁹` nên bit 1 ở vị trí 29 hợp lệ. **Sửa:** `L < 30` (hoặc `L < 31`, hoặc dùng `k.bit_length()`).

### Lỗi 3 — Đếm số 0 + số bit 1 của `k` (hoặc của `bin(k)`)

In [24]:
def wrong_popcount(s, k):
    return s.count("0") + min(s.count("1"), bin(k).count("1"))   # SAI: vị trí bit quan trọng

for s, k in [(s, k) for s, k, _ in examples] + [("0111", 4)]:
    print("bản sai :", wrong_popcount(s, k), "| bản đúng:", Solution().longestSubsequence(s, k), "cho", s, k)

bản sai : 6 | bản đúng: 5 cho 1001010 5
bản sai : 6 | bản đúng: 6 cho 00101001 1
bản sai : 2 | bản đúng: 3 cho 0111 4


**Vì sao sai:** ví dụ 1: bản sai cho `4 + min(3, 2) = 6`, nhưng mọi dãy con dài 6 đều > 5. Ví dụ 2 **trùng hợp** đúng (`5 + 1`). Số bit 1 nhận được phụ thuộc vào **vị trí** của chúng sau khi ghép với các số 0, không phải số bit 1 của `k`. `"0111"`, `k = 4`: nhận được `"011"` = 3 ⇒ dài 3; công thức sai cho `1 + 1 = 2`.

### Lỗi 4 — So sánh `< k` thay vì `≤ k`

In [25]:
def wrong_strict(s, k):
    ans = v = 0
    for c in reversed(s):
        if c == "0": ans += 1
        elif ans < 31 and v + (1 << ans) < k:       # SAI: được bằng k
            v += 1 << ans; ans += 1
    return ans

for s, k, _ in examples:
    print("bản sai :", wrong_strict(s, k), "| bản đúng:", Solution().longestSubsequence(s, k))

bản sai : 5 | bản đúng: 5
bản sai : 5 | bản đúng: 6


**Vì sao sai:** ví dụ 2: `k = 1`, bit 1 ở vị trí 0 cho đúng 1 = `k` — bản sai bỏ nó. Ví dụ 1 **trùng hợp** đúng.

### Lỗi 5 — Sau khi bỏ một bit 1, vẫn tiếp tục thử các bit 1 sau (không sai, nhưng dễ cài nhầm thành "trừ độ dài")

In [26]:
def wrong_decrement(s, k):
    ans = v = 0
    for c in reversed(s):
        if c == "0": ans += 1
        elif v + (1 << ans) <= k: v += 1 << ans; ans += 1
        else: ans -= 0 if ans == 0 else 1          # SAI: "bỏ" một ký tự đã nhận để chỗ cho bit 1?
    return ans

for s, k, _ in examples:
    print("bản sai :", wrong_decrement(s, k), "| bản đúng:", Solution().longestSubsequence(s, k))

bản sai : 3 | bản đúng: 5
bản sai : 4 | bản đúng: 6


**Vì sao sai:** khi bỏ một bit 1, ta không phải trả giá gì — chỉ đơn giản là không nhận nó. Giảm độ dài là bỏ một ký tự **đã nhận** hợp lệ, làm đáp án nhỏ đi. **Sửa:** nhánh `else` không làm gì.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Chuỗi nhị phân + ràng buộc giá trị ⇒ nghĩ theo **vị trí bit**: ký tự ở cuối có trọng số nhỏ nhất ⇒ xây **từ phải sang trái**.
- "Dãy con dài nhất có tổng/giá trị ≤ k" ⇒ phần tử "miễn phí" (số 0) luôn lấy; phần tử "tốn kém" lấy theo thứ tự rẻ nhất trước.
- `k ≤ 10⁹ < 2³⁰` ⇒ chỉ tối đa ~30 bit 1 có ý nghĩa; dùng `k.bit_length()` để tránh số ma thuật.
- Dạng đóng "hậu tố dài `m` hoặc `m − 1` + số 0" cho lời giải một dòng.
- Bài cùng họ: 1689 Partitioning Into Minimum Number of Deci-Binary Numbers, 1404, 1680 Concatenation of Consecutive Binary Numbers, 2275, 2546 Apply Bitwise Operations to Make Strings Equal, 1023.

## 15. Biến thể của bài

### 15.1 Biến thể A — Trả về chính dãy con (một nghiệm tối ưu, giá trị nhỏ nhất trong các nghiệm dài nhất)

In [27]:
def best_subsequence(s, k):
    L = v = 0; chosen = []
    for i in range(len(s) - 1, -1, -1):
        if s[i] == "0": chosen.append(i); L += 1
        elif L < 31 and v + (1 << L) <= k: v += 1 << L; chosen.append(i); L += 1
    return "".join(s[i] for i in sorted(chosen))

for s, k, exp in examples:
    t = best_subsequence(s, k); print(repr(t), int(t, 2), len(t) == exp)

'00010' 2 True
'000001' 1 True


### 15.2 Biến thể B — Ràng buộc **< k** (nhỏ hơn ngặt): thay `k` bằng `k − 1` (khi `k ≥ 1`)

In [28]:
def longest_less_than(s, k):
    if k == 0: return -1                         # không dãy con nào < 0 (chuỗi rỗng = 0)
    return Solution().longestSubsequence(s, k - 1) if k > 1 else s.count("0")

rng = random.Random(152)
for _ in range(500):
    n = rng.randint(1, 10); s = "".join(rng.choice("01") for _ in range(n)); k = rng.randint(1, 40)
    ref = max(L for L in range(n + 1) for idx in combinations(range(n), L)
              if (int("".join(s[i] for i in idx), 2) if L else 0) < k)
    assert longest_less_than(s, k) == ref
print(longest_less_than("1001010", 5), longest_less_than("00101001", 1))

5 5


Với `k = 1`, "< 1" nghĩa là giá trị 0 ⇒ chỉ lấy số 0 ⇒ `s.count("0")` (bản nộp với `k = 0` cũng cho đúng kết quả này nếu cho phép `k = 0`).

### 15.3 Biến thể C — Ràng buộc **≥ k** và tìm dãy con **ngắn nhất**

Đối ngẫu: dãy con ngắn nhất có giá trị `≥ k`. Chỉ cần bit 1 sớm nhất có đủ "chỗ" phía sau: lấy bit 1 tại vị trí `i` và `m − 1` ký tự bất kỳ sau nó cho giá trị `≥ 2^{m−1}`; nếu chưa đủ `≥ k` thì cần dài `m + 1`… — kiểm chứng bằng vét cạn:

In [29]:
def shortest_at_least(s, k):
    n, m = len(s), k.bit_length()
    best = inf
    for i in range(n):
        if s[i] != "1": continue
        # độ dài m: bit 1 tại i + m-1 ký tự sau; giá trị lớn nhất có thể = chọn các ký tự lớn nhất "kiểu tham lam"
        rest = s[i + 1:]
        if len(rest) >= m - 1:
            # chọn m-1 ký tự sau i để giá trị lớn nhất: tham lam chọn '1' sớm nhất còn đủ chỗ
            t = "1"; pos = 0
            for need in range(m - 1, 0, -1):
                window = rest[pos:len(rest) - need + 1]
                j = window.find("1"); j = 0 if j == -1 else j
                t += window[j]; pos += j + 1
            if int(t, 2) >= k: best = min(best, m)
            elif len(rest) >= m: best = min(best, m + 1)
        elif len(rest) >= 0 and False: pass
    return -1 if best == inf else best

def shortest_brute(s, k):
    n = len(s)
    for L in range(1, n + 1):
        if any(int("".join(s[i] for i in idx), 2) >= k for idx in combinations(range(n), L)): return L
    return -1

rng = random.Random(153)
for _ in range(500):
    n = rng.randint(1, 9); s = "".join(rng.choice("01") for _ in range(n)); k = rng.randint(1, 40)
    assert shortest_at_least(s, k) == shortest_brute(s, k), (s, k)
print(shortest_at_least("1001010", 5), shortest_at_least("00101001", 6))

3 3


Biến thể này khó hơn bản gốc một chút (cần chọn các ký tự **lớn nhất** sau bit 1 dẫn đầu) — đáng để so sánh: "≤ k dài nhất" thì chọn bit **rẻ nhất**, "≥ k ngắn nhất" thì chọn bit **đắt nhất**.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi dãy con | định nghĩa | O(2ⁿ·n) | O(n) | kiểm chứng |
| 1b. DP giá trị nhỏ nhất | theo độ dài | O(n²) | O(n) | tổng quát (ràng buộc khác) |
| 2. Tham lam từ phải | bit rẻ trước | O(n) | O(1) | **mặc định** |
| 3. Dạng đóng | `k.bit_length()` | O(n) | O(1) | một dòng |
| 4. Hậu tố + số 0 | tường minh | O(n) | O(1) | dễ chứng minh |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `longestSubsequence("111000", 4)` và `longestSubsequence("000", 1)`?

**Bài 2 (tính tay):** Trace tham lam cho `s = "101101"`, `k = 6`.

**Bài 3 (điền TODO):** Viết `bt3_todo(s, k)` cài **dạng đóng** theo `k.bit_length()`, sạch sẽ.

**Bài 4 (tự viết lại):** Viết `bt4_todo(s, k)` cài tham lam từ phải không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(s, k)` trả về **số lượng** dãy con (theo tập chỉ số) có độ dài bằng đáp án và giá trị ≤ `k`, với `n ≤ 60` và `k ≤ 1000` (gợi ý: DP theo (vị trí từ phải, độ dài, giá trị)).

In [30]:
def bt3_todo(s, k):
    # TODO: dạng đóng
    pass

def bt4_todo(s, k):
    # TODO: tham lam từ phải
    pass

def bt5_todo(s, k):
    # TODO: số dãy con dài nhất hợp lệ
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: ba số 0 cuối luôn lấy (L = 3). Mức 2: bit 1 tiếp theo ở vị trí 3: `8 > 4` bỏ, và các bit 1 sau cũng bỏ. Mức 3: `3`; câu hai: `3`.

**Bài 2.** Mức 1: từ phải: `1` (v = 1, L = 1), `0` (L = 2), `1` (1 + 4 = 5 ≤ 6, L = 3). Mức 2: `1` ở vị trí 3: `5 + 8 > 6` bỏ. Mức 3: `0` (L = 4), `1` bỏ ⇒ `4`.

**Bài 3.** Mức 1: `m = k.bit_length()`. Mức 2: nếu `n ≤ m` thì `n` hoặc `n − 1`. Mức 3: ngược lại: `zeros(s[:n−m]) + (m nếu int(s[n−m:]) ≤ k, ngược lại m − 1)`.

**Bài 4.** Mức 1: `L = v = 0`. Mức 2: số 0 ⇒ `L += 1`. Mức 3: số 1 ⇒ nhận nếu `v + 2^L ≤ k`.

**Bài 5.** Mức 1: biết độ dài mục tiêu `T` từ bản nộp. Mức 2: `cnt[L][v]` = số cách chọn dãy con (từ phải) dài `L` giá trị `v`. Mức 3: thêm ký tự `c` vào trước: `(L, v) → (L + 1, v + c·2^L)` nếu `≤ k`; đáp án `Σ_v cnt[T][v]`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `3`, `3`. **Bài 2:** `4`.

In [31]:
assert Solution().longestSubsequence("111000", 4) == 3
assert Solution().longestSubsequence("000", 1) == 3
assert Solution().longestSubsequence("101101", 6) == 4

def bt3(s, k):
    n, m = len(s), k.bit_length()
    if n <= m: return n if int(s, 2) <= k else n - 1
    return s[:n - m].count("0") + (m if int(s[n - m:], 2) <= k else m - 1)

def bt4(s, k):
    return greedy_right(s, k)

def bt5(s, k):
    T = Solution().longestSubsequence(s, k)
    cnt = {(0, 0): 1}
    for c in reversed(s):
        new = dict(cnt)
        for (L, v), w in cnt.items():
            nv = v + (int(c) << L) if L < 31 else (v if c == "0" else k + 1)
            if nv <= k: new[(L + 1, nv)] = new.get((L + 1, nv), 0) + w
        cnt = new
    return sum(w for (L, v), w in cnt.items() if L == T)

def bt5_brute(s, k):
    T = brute(s, k); n = len(s)
    return sum(1 for idx in combinations(range(n), T) if (int("".join(s[i] for i in idx), 2) if T else 0) <= k)

rng = random.Random(1995)
for _ in range(800):
    n = rng.randint(1, 11); s = "".join(rng.choice("01") for _ in range(n)); k = rng.randint(1, 300)
    assert bt3(s, k) == brute(s, k) and bt4(s, k) == brute(s, k)
    assert bt5(s, k) == bt5_brute(s, k)
print(bt3("1001010", 5), bt5("1001010", 5), bt5("00101001", 1))
print("Đáp án bài tập: tất cả assert OK")

5 3 1
Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
L = v = 0
for c in reversed(s):                 # xây từ phải: L = vị trí bit kế tiếp
    if c == "0":
        L += 1                        # số 0: miễn phí
    elif L < 31 and v + (1 << L) <= k:
        v += 1 << L; L += 1           # bit 1 rẻ nhất có thể
return L
```

**Checklist:**
- [ ] Duyệt **từ phải sang trái**.
- [ ] Luôn nhận số 0.
- [ ] `≤ k` (không phải `< k`).
- [ ] Giới hạn bit đủ rộng (`L < 30` hoặc `31`, hoặc theo `k.bit_length()`).